# 데이터 작성 배경
AI허브의 「경구약제 이미지 데이터」는 서울특별시 보라매병원이 주관해 구축했으며, 경희대학교 산학협력단과 여러 참여 기관이 약품 정보 정리, 이미지 수집·정제, 라벨링, 알고리즘 개발 등에 참여했습니다. 국내에서 자주 유통되는 경구약제를 다양한 각도와 조명, 배경에서 촬영하고 약품 정보와 바운딩박스 등을 표시해, 이미지로 약을 식별하는 AI 서비스 개발에 활용하도록 만든 데이터입니다. AI허브에 공개된 원본 데이터의 구축량은 총 2,793,978장입니다. 이번 프로젝트에서는 Codeit AI 15기 Kaggle 대회 데이터에서 제공한 정제본을 사용하고 있으며, 현재 확인한 학습 이미지는 232장입니다. 이 수는 AI허브 원본 전체가 아니라 프로젝트에서 사용하는 학습 세트의 이미지 수입니다. 원본에서 어떤 기준으로 선별하고 정제했는지는 Codeit AI 15기 Kaggle 대회 데이터에서 제공한 자료를 통해 확인할 예정입니다.

참고자료
- AI허브 [경구약제 이미지 데이터](https://aihub.or.kr/aihubdata/data/view.do?pageIndex=1&currMenu=115&topMenu=100&srchOptnCnd=OPTNCND001&searchKeyword=%EC%95%8C%EC%95%BD&srchDetailCnd=DETAILCND001&srchOrder=ORDER001&srchPagePer=20&aihubDataSe=data&dataSetSn=576), 2026년 10월 24일 확인.

# 알약 이미지 EDA
계산은 `src/eda.py`, 시각화는 `src/visualize.py`에 두고 노트북에서 호출합니다.
사진이 포함된 실행 출력은 팀 내부 검토용입니다. 공개 PR에는 출력 없는 노트북과 코드만 올립니다.

## 공통 준비 · 데이터 배경과 출처
사진 속 알약의 위치와 종류를 찾는 프로젝트입니다.
출처: [Codeit AI 15기 Kaggle 대회 데이터](https://www.kaggle.com/competitions/ai15-level1-project/data).
원본(v0)을 분석하며, 정제본·합성 데이터의 수치와 구분합니다.

**발견한 점:** 학습 이미지와 테스트 이미지는 역할이 다릅니다.

**다음 행동:** 원본은 보존하고, 수정은 별도 데이터 버전으로 관리합니다.

### EDA란?
모델에게 공부시킬 자료를 먼저 펼쳐 보는 작업입니다. 자료량, 정답 표시, 촬영 조건을 확인합니다.

**발견한 점:** 사진 수만으로는 자료 품질이나 모델 성능을 알 수 없습니다.

**다음 행동:** 통계에서 발견한 후보와 사진으로 확인한 사실을 구분합니다.

### 데이터 읽기
설정 파일의 상대 경로를 사용합니다. 프로젝트 루트 또는 notebooks 폴더에서 실행합니다.
`data/raw/` 아래에 원본을 연결하고 모든 셀을 위에서부터 실행하세요.

**발견한 점:** 원본 사진 232장·JSON 763개·클래스 56개·박스 763개입니다.

**다음 행동:** 수치가 다르면 코드보다 데이터 경로와 버전을 먼저 확인합니다.

## 이 노트북을 읽는 순서

공통 준비 다음에 D1~D11 순서로 분석을 정리했습니다. 기존 통계·시각화는 보존하고, 근거가 부족한 단계에는 필요한 분석을 적었습니다. 제목의 D 번호는 실험 분류이며 해당 실험을 모두 수행했다는 뜻은 아닙니다.

**수치 해석:** 원본 전체의 클래스 수·박스 수와 학습 분할의 수치는 구분합니다. 검증 정답 교정 전후의 점수도 같은 기준으로 섞어 비교하지 않습니다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("프로젝트 폴더에서 실행하세요.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import eda as eda
from src import visualize as viz

viz.set_korean_font()
CONFIG = "configs/eda_soonwoo.yaml"
ctx = eda.load_eda_context(CONFIG)
display(eda.eda_inventory(ctx))

### 원본 사진 둘러보기
10행 × 5열, seed 42로 같은 사진을 선택합니다.

**발견한 점:** 한 장에 여러 알약이 등장합니다. 무작위 50장은 전체 품질을 보증하는 검사가 아닙니다.

**다음 행동:** 아래 클래스별 사진과 오류 후보를 별도로 확인합니다.

In [ ]:
viz.plot_random_photos(ctx["image_dir"], n=50, rows=10)

### 파일명과 메타데이터 읽기
파일명 토큰과 JSON 메타데이터를 나란히 표시합니다.

**발견한 점:** 앞쪽 알약 코드와 각도 토큰을 대조할 수 있습니다. 배경·조명 토큰 전체의 공식 코드표는 미확보입니다.

**다음 행동:** 확인되지 않은 토큰에 색상·촬영 의미를 임의로 붙이지 않습니다.

In [ ]:
display(eda.filename_token_table(ctx))
display(eda.filename_metadata(ctx))

간단하게 정리하면 이렇게 설명 가능합니다.


| 토큰 위치 |	예시 값	| 매칭되는 메타데이터 항목 |	분석 상태 |
| --- | --- | --- | --- |
| 0번째	| 0	| drug_dir (알약의 방향: 앞면/뒷면 등) |	대조 필요 (추정 중) |
| 1번째	| 2	| light_color (조명 색상: 주백색 등) |	대조 필요 (추정 중) |
| 2번째	| 0	| back_color (배경 색상: 연회색 등) |	대조 필요 (추정 중) |
| 3번째	| 2	| 알 수 없음	미확정| |
| 4번째	| 70	| camera_la (카메라 고도 / Latitude) |	매칭 확실 (대조 완료) |
| 5번째	| 000	| camera_lo (카메라 방위각 / Longitude)	| 대조 필요·의미 미확인 |
| 6번째	| 200	| size (이미지 또는 알약 크기)	| 대조 필요·의미 미확인 |


## D1 · 희귀 알약 반복 학습

- **확인할 것:** 어떤 알약의 학습 자료가 부족한가?
- **가설:** 학습셋에서 드물게 등장하는 알약 사진을 더 자주 보여주면 해당 클래스 성능이 개선될 수 있습니다.
- **추가로 필요한 것:** 학습 분할만으로 센 클래스별 사진·박스 수, RFS 적용 전후 노출 수, 희귀 클래스별 검증 성능. 아래 전체 원본 분포와 구분해야 합니다.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 클래스별 자료량 확인
알약 종류별 정답 박스 수를 셉니다.

**발견한 점:** 원본에서 클래스별 박스 수는 최소 3개·최대 153개로 차이가 큽니다.

**다음 행동:** 드문 클래스를 더 자주 학습하는 방법을 같은 검증 분할에서 비교합니다.

In [ ]:
class_counts = eda.eda_class_counts(ctx)
display(class_counts)
viz.plot_eda_counts(class_counts, "label_idx", "박스 수", "라벨별 정답 박스 수")

## D2 · 희귀 알약 합성

- **확인할 것:** 희귀 알약을 잘라 붙일 원본과 빈 배경이 충분한가?
- **가설:** 희귀 알약을 다른 학습 사진의 빈 위치에 합성하면 학습 상황을 늘릴 수 있습니다.
- **추가로 필요한 것:** 학습셋 전용 합성 원본·배경, 합성 전후 클래스별 박스 수, 경계·겹침·조명 자연스러움 검수, 원본 검증셋 유지 여부.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 합성 후보 알약과 라벨 확인
정답 박스로 잘라 연속 라벨·원본 코드·이름을 함께 표시합니다.

**발견한 점:** category_id와 label_idx는 서로 다른 번호입니다. 대표 사진만으로 모든 라벨의 정확성을 확정할 수 없습니다.

**다음 행동:** 모양이 다른 사례는 원본 사진과 JSON까지 함께 검토합니다.

In [ ]:
viz.plot_class_crops(ctx)

### D2 · 아직 필요한 분석

대표 알약 사진만으로 합성이 자연스럽다고 판단할 수 없습니다. 실제 copy-paste 전후 예시와 생성 로그를 추가하고, 검증 사진이 합성 재료에 포함되지 않았는지 확인해야 합니다.


## D3 · 안전한 이미지 증강

- **확인할 것:** 촬영 각도·배경·조명 변화에 대응할 필요가 있는가?
- **가설:** 알약 종류를 구분하는 단서를 유지하면서 반전·채도·명도·크기를 바꾸면 촬영 변화에 대응할 수 있습니다.
- **추가로 필요한 것:** 실제 학습 YAML과 증강된 이미지, 박스 좌표 동반 변환, 촬영 조건별 성능 비교.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### D3 · 실제 설정과 설명 맞추기

팀 YOLO 안전 증강 설정(`yolo_aug_safe.yaml`)은 좌우·상하 반전, 채도·명도·크기·이동 및 Mosaic을 사용하고 `degrees: 0.0`, `hsv_h: 0.0`으로 둡니다. 90도 회전을 적용했다고 표현하지 않습니다. 실제 실행에 사용한 설정도 확인해야 합니다. 아래 반전 그림은 동작 예시이며 전체 증강 조합의 효과를 입증하는 자료는 아닙니다.


### 촬영 배경·조명 분포
배경·조명 메타데이터별 고유 사진 수입니다.

**발견한 점:** 메타데이터 코드별 수량은 확인되지만 실제 배경색을 보증하지 않습니다.

**다음 행동:** 성능과 배경의 관계를 볼 때 사진과 함께 확인합니다.

In [ ]:
background = eda.metadata_counts(ctx, "back_color")
display(background)
display(eda.metadata_counts(ctx, "light_color"))
viz.plot_eda_counts(background, "back_color", "고유 사진 수", "배경 코드별 고유 사진 수")

### 촬영 각도 분포
JSON 각도별 사진 수와 파일명 각도 일치를 봅니다.

**발견한 점:** 촬영 각도별 표본 수와 코드 일치 여부를 확인할 수 있습니다. 각도 토큰이 사진 품질을 의미하지는 않습니다.

**다음 행동:** 각도별 오류율은 모델 예측을 받은 뒤 같은 기준으로 계산합니다.

In [ ]:
angles = eda.metadata_counts(ctx, "camera_la")
display(angles)
display(eda.angle_alignment(ctx))
viz.plot_eda_counts(angles, "camera_la", "고유 사진 수", "촬영 각도별 고유 사진 수")

### 증강 전후 박스 확인
좌우 반전 시 박스 좌표도 함께 변환합니다.

**발견한 점:** 이미지만 반전하면 기존 박스 좌표가 알약 위치와 어긋납니다. 아래 그림은 동작 예시이며 성능 개선 증거는 아닙니다.

**다음 행동:** 학습용 증강 설정을 바꿀 때 박스·라벨의 동반 변환부터 확인합니다.

In [ ]:
viz.plot_flip_example(ctx)

## D4 · 고해상도 학습

- **확인할 것:** 작은 알약의 특징이 입력 축소로 사라질 수 있는가?
- **가설:** 640보다 높은 해상도로 학습하면 작은 알약의 시각 정보를 더 보존할 수 있습니다.
- **추가로 필요한 것:** 640·1024 축소 이미지의 알약·각인 확대 비교, 작은 박스별 성능, GPU 메모리·학습 및 추론 시간.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 이미지 크기와 상대 박스 크기
크기·알약 수·상대 박스 크기·파일 무결성을 확인합니다.

**발견한 점:** 원본 읽기 오류·메타데이터 크기 불일치·동일 바이트 중복 그룹은 0입니다. 작은 박스는 해상도 실험의 근거가 됩니다.

**다음 행동:** 경계 밖·겹침 후보는 사진으로 확인하고 자동 삭제하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

INPUT_SIZES = (640, 1024)

# JSON 크기만 믿지 않고 실제 이미지 파일의 크기를 확인
image_rows = []

for path in ctx["paths"]:
    with Image.open(path) as image:
        width, height = image.size

    image_rows.append(
        {
            "file_name": path.name,
            "original_width": width,
            "original_height": height,
        }
    )

image_sizes = pd.DataFrame(image_rows)

if image_sizes.empty:
    raise ValueError("이미지가 없습니다. ctx의 데이터 경로를 확인해주세요.")

print("원본 이미지 해상도별 사진 수")
resolution_counts = (
    image_sizes.groupby(["original_width", "original_height"])
    .size()
    .reset_index(name="사진 수")
    .sort_values("사진 수", ascending=False)
)
display(resolution_counts)

# 실제 이미지 크기와 알약 박스를 파일명으로 연결
size_boxes = ctx["boxes"].merge(
    image_sizes,
    on="file_name",
    how="left",
    validate="many_to_one",
)

if size_boxes.empty:
    raise ValueError("분석할 정답 박스가 없습니다.")

if size_boxes[["original_width", "original_height"]].isna().any().any():
    raise ValueError("일부 정답 박스의 이미지 파일을 찾지 못했습니다.")

if (size_boxes[["box_w", "box_h"]] <= 0).any().any():
    raise ValueError("너비 또는 높이가 0 이하인 박스를 먼저 검토해주세요.")

long_side = size_boxes[["original_width", "original_height"]].max(axis=1)

summary_rows = []

for size in INPUT_SIZES:
    # 비율 유지 리사이즈의 근사값. 여백은 박스 크기를 바꾸지 않는다.
    ratio = size / long_side

    size_boxes[f"width_{size}"] = size_boxes["box_w"] * ratio
    size_boxes[f"height_{size}"] = size_boxes["box_h"] * ratio
    size_boxes[f"short_{size}"] = size_boxes[[f"width_{size}", f"height_{size}"]].min(axis=1)

    short_side = size_boxes[f"short_{size}"]

    summary_rows.append(
        {
            "입력 크기": size,
            "박스 수": len(short_side),
            "짧은 변 최소(px)": short_side.min(),
            "짧은 변 하위 10%(px)": short_side.quantile(0.1),
            "짧은 변 중앙값(px)": short_side.median(),
            # 아래 기준은 분포 관찰용이며 검출 성공·실패 기준이 아니다.
            "짧은 변 16px 미만(%)": short_side.lt(16).mean() * 100,
            "짧은 변 32px 미만(%)": short_side.lt(32).mean() * 100,
        }
    )

print("\n입력 크기별 알약 박스 크기")
resolution_summary = pd.DataFrame(summary_rows)
display(resolution_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(
    image_sizes["original_width"],
    image_sizes["original_height"],
    alpha=0.4,
)
axes[0].set(
    title="Original image dimensions",
    xlabel="Width (px)",
    ylabel="Height (px)",
)

for size in INPUT_SIZES:
    axes[1].hist(
        size_boxes[f"short_{size}"],
        bins=25,
        alpha=0.5,
        label=f"Input {size}",
    )

axes[1].set(
    title="Pill box short side after resizing",
    xlabel="Short side (px)",
    ylabel="Number of boxes",
)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n640에서 가장 작게 표현되는 알약 10개")
display(
    size_boxes.sort_values("short_640")[
        [
            "file_name",
            "name",
            "box_w",
            "box_h",
            "short_640",
            "short_1024",
        ]
    ]
    .head(10)
    .round(2)
)

파란색 (640 모델 입력 시): 알약의 짧은 변이 대부분 70px ~ 120px 사이로 아주 작아집니다. 심지어 50px대에 가까운 극단적으로 작은 박스들도 생깁니다.
<br>주황색 (1024 모델 입력 시): 알약의 짧은 변이 대부분 130px ~ 200px 사이로 어느 정도 크기를 유지합니다.

In [ ]:
import math

# 극단적으로 작은 박스가 오류일 수 있으므로 원본도 함께 확인
examples = size_boxes.sort_values("short_640").head(3)
fig, axes = plt.subplots(
    len(examples),
    3,
    figsize=(12, 4 * len(examples)),
    squeeze=False,
)

for row_index, (_, box) in enumerate(examples.iterrows()):
    image_path = ctx["image_dir"] / box["file_name"]

    with Image.open(image_path) as image:
        original = image.convert("RGB")

    width, height = original.size

    for column, target in enumerate([None, *INPUT_SIZES]):
        if target is None:
            resized = original
            title = "Original"
        else:
            ratio = target / max(width, height)
            resized = original.resize(
                (
                    max(1, round(width * ratio)),
                    max(1, round(height * ratio)),
                ),
                Image.Resampling.BILINEAR,
            )
            title = f"Input {target}"

        # 丸め後の実際の画像サイズで座標を合わせる
        scale_x = resized.width / width
        scale_y = resized.height / height

        left = max(0, math.floor(box["x1"] * scale_x))
        top = max(0, math.floor(box["y1"] * scale_y))
        right = min(resized.width, math.ceil(box["x2"] * scale_x))
        bottom = min(resized.height, math.ceil(box["y2"] * scale_y))

        if right <= left or bottom <= top:
            raise ValueError(f"画像範囲外のボックス: {box['file_name']}")

        crop = resized.crop((left, top, right, bottom))

        axes[row_index, column].imshow(crop, interpolation="nearest")
        axes[row_index, column].set_title(
            f"{title}: {crop.width} × {crop.height} px\n{box['name']}"
        )
        axes[row_index, column].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
display(eda.image_integrity(ctx))
display(eda.image_size_stats(ctx["images"]))
display(eda.objects_per_image(ctx["images"]))
display(eda.bbox_stats(ctx["boxes"]))
display(eda.class_box_size(ctx["boxes"]).head(10))
viz.plot_hist(ctx["boxes"]["rel_area"], "이미지 대비 박스 면적", xlabel="rel_area")
viz.plot_box_centers(ctx["boxes"])

### D4 · 해상도 가설을 보완하려면

작은 박스가 있다는 통계만으로 1024 입력이 더 좋다고 결론 내릴 수 없습니다. 같은 분할·모델·학습 조건으로 해상도만 바꿔 비교하고, 각인이 실제로 판독 가능해지는지도 사진으로 확인해야 합니다.


## D5 · 잘못된 데이터 제거

- **확인할 것:** 파일이나 정답에 학습을 방해하는 오류가 있는가?
- **가설:** 사람이 확인한 심한 오류를 학습셋에서 제외하면 잘못된 정답의 영향을 줄일 수 있습니다.
- **추가로 필요한 것:** 전체 검수 범위, 제외 목록·사유, 제외 전후 클래스별 수량. 자동 후보를 확정 오류로 취급하지 않습니다.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 사진과 JSON의 파일 대조
사진 목록과 JSON 이미지 목록을 양방향으로 비교합니다.

**발견한 점:** 원본에서 사진만 있거나 JSON만 있는 파일은 각각 0개입니다. 일부 알약의 박스 누락 8건과는 다른 검사입니다.

**다음 행동:** 새 버전에서도 파일 대조와 박스 대조를 각각 실행합니다.

In [ ]:
display(eda.file_alignment(ctx))

### 누락·경계·겹침 오류 후보
파일명 차집합과 박스 경계·겹침 등을 검사합니다.

**발견한 점:** 원본에서 누락 후보 사진 8장이 나옵니다. 10/26 사진 대조에서는 각 사진에 박스 없는 알약 1개가 확인됐습니다.

**다음 행동:** 누락 코드의 약품 정체는 파일명 추정과 구분하고, 정제본 반영 여부는 별도로 확인합니다.

In [ ]:
missing = eda.filename_missing_candidates(ctx)
display(missing)
display(eda.find_label_issues(ctx["images"], ctx["boxes"], image_dir=ctx["image_dir"]))
viz.plot_missing_candidates(ctx, missing)

### D5 · 제외와 교정을 구분하기

누락된 정답을 단순히 제외하는 것과, 정답 박스·종류를 바로잡는 것은 다른 처리입니다. `manual_drop` 적용 여부와 제외 단위(객체 또는 이미지)는 실제 코드·기록으로 확인합니다. 종류 변경·박스 추가의 사진 근거는 D10에서 다룹니다.


## D6 · 외부 데이터 추가

- **확인할 것:** 현재 자료로 부족한 촬영 상황을 외부 데이터가 보완하는가?
- **가설:** 원본과 관련된 외부 학습 자료를 추가하면 부족한 클래스와 촬영 상황을 보완할 수 있습니다.
- **추가로 필요한 것:** 출처·사용 조건·대회 허용 범위, 원본과 외부의 클래스 대응표, 촬영 조건 및 클래스 분포, 원본 검증·테스트와의 중복 확인.

팀 테스크에 맞춰 학습 후, EDA를 실시하여 사용 가능여부를 판단할 예정이며, EDA는 실험의 근거를 찾는 과정이고 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### D6 · 추가 분석 필요

외부 이미지·라벨과 출처 목록을 준비한 뒤 원본과 수량·크기·배경·각도·클래스 분포를 나란히 비교해야 합니다. 자료가 더 많다는 이유만으로 효과를 확정하지 않습니다.


## D7 · 여러 분할로 재검증

- **확인할 것:** 사진을 나누는 방식에 따라 점수가 달라지는가?
- **가설:** 여러 분할의 점수를 비교하면 특정 검증셋에 따른 변동을 파악할 수 있습니다.
- **추가로 필요한 것:** 5개 fold별 클래스·사진 수, 조합 그룹 중복 여부, 희귀 클래스 포함 여부, 동일 지표의 평균·표준편차. K-fold 평균만으로 실사용 성능을 보증하지 않습니다.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 같은 알약 조합의 촬영 사진 확인
파일명 기준 같은 알약 조합의 사진 수를 셉니다.

**발견한 점:** 232장에는 같은 조합의 다른 각도 사진이 포함되며, 파일명 기준 조합은 114개입니다.

**다음 행동:** 같은 조합이 학습·검증 양쪽에 겹치지 않도록 그룹 분할도 비교합니다.

In [ ]:
combinations = eda.pill_combinations(ctx)
display(combinations.head(20))
print("고유 조합 수:", len(combinations))

### D7 · 분할 가설을 보완하려면

같은 조합의 각도만 다른 사진이 학습·검증에 함께 있으면 평가가 쉬워질 수 있습니다. 일반 분할과 조합 단위 그룹 분할을 구분하고, fold별 목록과 성능 기록을 추가해야 합니다. 현재 조합 통계는 K-fold 수행 결과가 아닙니다.


## D8 · 모델 예측을 재학습에 활용

- **확인할 것:** 라벨 없는 자료를 신뢰할 만한 임시 정답으로 활용할 수 있는가?
- **가설:** 검수한 고신뢰도 예측이 유용한 학습 자료가 될 수 있지만, 잘못된 예측을 반복 학습할 위험도 있습니다.
- **추가로 필요한 것:** 대회 규칙상 테스트 의사 라벨 사용 허용 여부, 독립 검증셋, 신뢰도 기준·클래스별 선별 수·사람 검수, 적용 전후 성능.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 테스트 이미지 현황 확인
정답 없는 테스트 사진을 학습과 별도로 검사합니다.

**발견한 점:** 테스트 842장의 읽기 오류는 0개입니다. 정답이 없어 이 분석만으로 정확도를 계산할 수 없습니다.

**다음 행동:** 테스트 사진을 학습 증강에 섞지 않고 최종 평가에 사용합니다.

In [ ]:
test_inventory = eda.test_image_inventory(ctx)
display(test_inventory)
viz.plot_random_photos(ctx["test_dir"], n=10)

### D8 · 테스트 활용에 대한 원칙

기존 분석은 테스트 이미지를 최종 제출 예측용으로 분리하는 원칙을 따릅니다. D8은 그 원칙을 바꾸는 별도 실험이므로 규칙상 허용 여부와 팀 합의를 먼저 확인해야 합니다. 테스트 사진 수·무결성만으로 의사 라벨의 정확도를 판단할 수 없으며, 테스트 정답이 없으므로 여기서 정확도를 계산하지 않습니다.


## D9 · 더 강한 증강

- **확인할 것:** D3보다 강한 변화가 유용한가, 식별 단서를 훼손하는가?
- **가설:** MixUp이나 더 넓은 크기 변화가 일반화를 도울 수 있지만 알약 색·각인을 흐릴 수도 있습니다.
- **추가로 필요한 것:** D3·D9 실제 YAML 차이, 혼합 비율별 예시, 각인·색 구분 가능성, 박스·라벨 정확성, 같은 검증셋의 성능.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### D9 · 추가 분석 필요

현재 노트북에는 MixUp 결과나 강도별 비교 그림이 없습니다. 약한·강한 증강을 같은 원본 사진에 적용해 사람이 식별 가능한지 확인하고, 모델 실험으로 효과를 검증해야 합니다. D3의 반전 예시만으로 더 강한 증강의 필요성을 입증할 수 없습니다.


## D10 · 틀린 라벨 직접 수정

- **확인할 것:** 모델 오류처럼 보이는 사례가 실제로 정답 오류인가?
- **가설:** 사진 근거로 클래스·박스를 교정하면 데이터를 보존하면서 학습·평가 정답의 신뢰성을 높일 수 있습니다.
- **추가로 필요한 것:** 교정 전후 사진·참고 사진, manual_fix 기록, 검수자·판정 근거, 정답 버전, 고정 예측의 교정 전후 재평가.

아래 EDA는 실험의 근거를 찾는 과정이며, 성능 개선 여부는 같은 검증 기준의 모델 실험으로 확인합니다.


### 사진으로 확인한 종류 라벨 오류 3건
같은 사진의 각인과, 다른 사진의 같은 라벨 알약을 비교한 뒤 팀 `manual_fix.csv`와 대조했습니다.

**발견한 점:** 플라빅스→일양하이트린 1건과 아토젯↔트윈스타 2건은 이미 교정표에 등록돼 있습니다. 두 모델에서 공통으로 오분류처럼 집계된 사례입니다. 사진의 약품을 새로 감정한 결과가 아니라 데이터의 라벨·교정표 대조 결과입니다.

**다음 행동:** 데이터 정답 버전을 맞춘 뒤 모델 오류를 해석합니다. 아래 사진은 왼쪽부터 문제 영역, 원래 정답 이름의 참고 사진, 예측·교정 이름의 참고 사진입니다.


In [ ]:
quality_audit = eda.load_quality_audit(ctx, CONFIG)
display(eda.quality_audit_table(quality_audit, "labels"))
viz.plot_quality_audit(quality_audit, ctx, "labels")

### 교정표의 누락 박스 추가 4건
초록은 기존 정답 박스, 주황은 교정표에서 추가한 정답 박스입니다. 왼쪽은 교정 전, 오른쪽은 추가 후입니다.

**발견한 점:** 이번 검증 47장에 해당하는 교정표에는 종류 변경 3건과 박스 추가 4건이 있습니다. 박스 수는 153개에서 157개가 됩니다. 앞서 찾은 누락 후보 8장 전부를 해결했다는 뜻은 아닙니다.

**다음 행동:** 나머지 누락 후보의 처리·제외 여부를 별도로 확인합니다. 원본(v0)은 보존하고 교정본과 구분합니다.


In [ ]:
display(eda.quality_audit_table(quality_audit, "missing"))
viz.plot_quality_audit(quality_audit, ctx, "missing")

### 정답 교정이 모델 평가에 미친 영향
**EDA에서 확인한 것:** 사진과 정답 라벨·교정표의 불일치입니다.

**모델 실험에서 확인한 것:** 저장된 예측을 같은 검증 47장의 교정 정답으로 다시 채점했습니다. 모델을 재학습하거나 새 예측을 만든 것은 아닙니다. 교정 전 점수가 원래 기록과 재현되는지 먼저 확인했습니다.

D1의 mAP@[0.75:0.95]는 0.96903→0.98341, D1+증강은 0.93817→0.95234입니다. D3 실행에는 D1의 RFS가 유지돼 순수 D3와 구분합니다. manual_fix만 적용했으며 manual_drop·자동 정제를 포함한 전체 v1 평가는 아닙니다.

**다음 행동:** 교정 정답을 공통 기준으로 정하고, 모델이 남긴 실제 혼동과 라벨 오류를 구분합니다. 이 점수는 검증 점수이며 캐글 점수·정답률이 아닙니다.

[팀 W&B의 원래 점수·교정 점수·비교 자료](https://forge.coreweave.com/wandb/YacssoK/soonwoo_yolov8m)

집계 점수는 2026-11-02 고정 기록입니다. 교정표가 바뀌면 사진 대조 표와 집계 시점이 달라질 수 있으므로 새 정답으로 재평가해야 합니다.


In [ ]:
display(eda.quality_audit_table(quality_audit, "models"))
viz.plot_quality_audit(quality_audit, ctx, "models")

## D11 · 알약 글자 읽기

- **확인할 것:** 색과 모양이 비슷한 알약을 각인으로 구분할 수 있는가?
- **가설:** 구별 가능한 각인 문자가 있다면 OCR 결과를 추가 단서로 사용할 수 있습니다.
- **추가로 필요한 것:** 실제 혼동 클래스 쌍, 각인 확대 사진과 문자 정답, 해상도·방향·반사별 판독 가능성, OCR 정확도와 탐지 결합 성능.

### D11 · 추가 분석 필요

팀 테스크에 따라서 외부데이터 확인 전, 모델 성능 비교 평가 후, 혼동행렬로 확인 시에 필요하다면 추가 분석을 통해 실시합니다. D2의 클래스별 잘라낸 사진과 D10의 교정 사진은 검토 출발점으로 활용할 수 있지만 OCR 성능의 근거는 아닙니다. 따라서 모델 성능 비교 시, 각인이 안 보이거나 문자가 비슷한 사례를 포함해 확인하고, 글자를 읽지 못할 때의 처리 기준도 정해야 합니다.

## 전체 정리 · 관찰과 실험 결과 구분
원본 분석 결과를 다시 모읍니다.

**발견한 점:** 클래스 불균형과 원본 라벨 누락이 있고, 사진·JSON 파일 자체의 짝은 맞습니다.

**다음 행동:** 정제본 처리 여부와 검증 분할을 확인한 뒤 증강·모델 실험 효과를 비교합니다.

In [ ]:
display(eda.eda_inventory(ctx))
display(eda.file_alignment(ctx))

# 앞선 분석 셀을 건너뛰어도 요약할 수 있도록 직접 계산
summary_missing = eda.filename_missing_candidates(ctx)
summary_test = eda.test_image_inventory(ctx)

print(
    "누락 후보 사진:",
    len(summary_missing),
    "/ 테스트 사진:",
    int(summary_test["사진 수"].sum()),
)

바운딩 박스 이상을 포함한 전체 결측치 확인을 실시합니다.

In [ ]:
import ast
import csv
import textwrap
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

from src.utils import load_config, resolve_path

# 앞선 분석 셀의 변수에 의존하지 않고 다시 계산
missing_candidates = eda.filename_missing_candidates(ctx)
geometry_candidates = eda.find_label_issues(ctx["images"], ctx["boxes"], image_dir=ctx["image_dir"])

records_by_name = {record["file_name"]: record for record in ctx["records"]}
issues_by_name = defaultdict(list)

issue_names = {
    "out_of_bounds": "박스가 이미지 경계 밖으로 나감",
    "tiny_or_empty": "박스 크기가 지나치게 작음",
    "too_many": "객체 수가 4개를 초과함",
    "no_boxes": "정답 박스가 없음",
    "heavy_overlap": "박스가 크게 겹침 — 오류 여부 검토 필요",
    "missing_file": "이미지 파일이 없음",
}

for _, row in missing_candidates.iterrows():
    codes = ast.literal_eval(row["파일명 기준 누락 코드"])
    issues_by_name[row["file_name"]].append(
        {
            "유형": "정답 누락 후보",
            "설명": f"파일명에는 있으나 정답에 없는 코드: {codes}",
            "대상 코드": None,
        }
    )

for _, row in geometry_candidates.iterrows():
    category = None if pd.isna(row["category_id"]) else int(row["category_id"])
    issues_by_name[row["file_name"]].append(
        {
            "유형": issue_names.get(row["issue"], row["issue"]),
            "설명": (
                f"대상 코드: {category}"
                if category is not None
                else "사진과 원본 JSON을 함께 확인해야 함"
            ),
            "대상 코드": category,
        }
    )

# 교정표가 있으면 표시용으로 읽기만 한다.
fixes_by_name = defaultdict(list)
config = load_config(CONFIG)
fix_value = config.get("eda_audit", {}).get("manual_fix")

if fix_value:
    fix_path = resolve_path(fix_value)
    if fix_path.is_file():
        with fix_path.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                fixes_by_name[row["file_name"]].append(row)

review_rows = []
for file_name, items in sorted(issues_by_name.items()):
    for item in items:
        review_rows.append(
            {
                "파일명": file_name,
                "유형": item["유형"],
                "설명": item["설명"],
                "교정표 기록": ("있음" if fixes_by_name[file_name] else "없음"),
            }
        )

review_table = pd.DataFrame(
    review_rows,
    columns=["파일명", "유형", "설명", "교정표 기록"],
)

print("누락 후보 사진:", len(missing_candidates))
print("박스 검사 후보 건수:", len(geometry_candidates))
print("중복을 제외한 사진:", len(issues_by_name))
display(review_table)

if not issues_by_name:
    print("발견된 후보 없음 — 현재 자동 검사 기준에 해당하는 항목이 없습니다.")

save_dir = resolve_path("reports/figures/eda_label_review")
save_dir.mkdir(parents=True, exist_ok=True)

review_table.to_csv(
    save_dir / "label_issue_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)

file_names = sorted(issues_by_name)

# 12장을 넘으면 다음 페이지에 이어서 표시
for start in range(0, len(file_names), 12):
    fig, axes = plt.subplots(4, 3, figsize=(21, 28))

    for ax in axes.ravel():
        ax.axis("off")

    for ax, file_name in zip(axes.ravel(), file_names[start : start + 12]):
        items = issues_by_name[file_name]
        fixes = fixes_by_name[file_name]
        record = records_by_name.get(file_name)
        image_path = ctx["image_dir"] / file_name

        descriptions = [f"{item['유형']}\n{item['설명']}" for item in items]
        status = "교정표 기록 있음 — 아래 내용 확인" if fixes else "후보 상태 — 사람의 확인 필요"
        title = (
            textwrap.fill(file_name, width=42)
            + "\n"
            + "\n".join(descriptions)
            + "\n상태: "
            + status
        )
        ax.set_title(title, fontsize=10, pad=10)

        if not image_path.is_file():
            ax.text(
                0.5,
                0.5,
                "이미지 파일 없음",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            continue

        with Image.open(image_path) as image:
            ax.imshow(image.convert("RGB"))

        suspect_codes = {item["대상 코드"] for item in items if item["대상 코드"] is not None}

        if record is not None:
            for box, category in zip(record["boxes"], record["category_ids"], strict=True):
                x1, y1, x2, y2 = box
                color = "red" if category in suspect_codes else "limegreen"
                ax.add_patch(
                    Rectangle(
                        (x1, y1),
                        x2 - x1,
                        y2 - y1,
                        fill=False,
                        edgecolor=color,
                        linewidth=2,
                    )
                )
                ax.text(
                    x1,
                    max(0, y1 - 5),
                    str(category),
                    fontsize=8,
                    color="black",
                    bbox={
                        "facecolor": color,
                        "alpha": 0.8,
                        "pad": 1,
                    },
                )

        fix_notes = []
        for fix in fixes:
            category = fix["category_id"]

            if fix["action"] == "add":
                coordinates = [float(value) for value in fix["new_value"].split()]
                if len(coordinates) != 4:
                    raise ValueError(f"교정표의 xyxy 좌표를 확인하세요: {file_name}")

                x1, y1, x2, y2 = coordinates
                ax.add_patch(
                    Rectangle(
                        (x1, y1),
                        x2 - x1,
                        y2 - y1,
                        fill=False,
                        edgecolor="orange",
                        linewidth=2.5,
                        linestyle="--",
                    )
                )
                fix_notes.append(f"교정표: 박스 추가 / 코드 {category}")

            elif fix["action"] == "class":
                fix_notes.append(f"교정표: 클래스 {category} → {fix['new_value']}")
            else:
                fix_notes.append(f"교정표: {fix['action']} / 코드 {category}")

        if fix_notes:
            ax.text(
                0.02,
                0.02,
                "\n".join(fix_notes),
                transform=ax.transAxes,
                fontsize=9,
                va="bottom",
                bbox={
                    "facecolor": "white",
                    "alpha": 0.9,
                    "pad": 3,
                },
            )

    fig.suptitle(
        "정답 라벨 품질 검토: 누락·경계·겹침 후보\n"
        "초록: 기존 정답 / 빨강: 코드로 위치를 특정한 의심 박스 / "
        "주황 점선: 교정표의 추가 박스\n"
        "좌표 없는 누락 위치는 추정하지 않음 · 겹침은 오류 확정이 아님",
        fontsize=15,
        y=0.995,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    page_number = start // 12 + 1
    save_path = save_dir / f"label_review_4x3_page{page_number}.png"
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("이미지 저장:", save_path)

print("후보 목록 저장:", save_dir / "label_issue_candidates.csv")

빨강: 확인이 필요한 박스.

### 결론 — 파일 연결과 정답 품질은 별도로 검토해야 한다

사진과 JSON 이미지 기록의 연결 누락은 없었다.
그러나 파일명 기준 정답 누락 후보 8장과,
경계 밖 박스 1장·높은 겹침 후보 3장이 확인됐다.

이는 오류 확정 수가 아니라 검토 대상 수다.
겹치는 알약은 정상일 수 있으므로 사진과 JSON을 함께 확인해야 한다.

처리 방향:
- 누락 확인: 객체의 정답 박스 추가
- 위치·크기 오류 확인: 박스 좌표 교정
- 클래스 오류 확인: 근거를 대조한 뒤 라벨 교정
- 정상 겹침: 기존 정답 유지
- 판단 곤란: 검토 보류 또는 제외 여부 결정

원본은 보존하고, 교정 사항을 별도 표에 기록한다.
교정 전후 성능을 비교할 때는 정답 버전과 검증 분할을 명시한다.